# NASA MSL Unconstrained Cache Generation: Ultra-VETime Telemetry-Command Teacher
### High-Density Curriculum Caching & Dedicated Head-to-Head Evaluation Cache
- **Domain**: NASA Mars Science Laboratory Rover (MSL, 73,729 steps, 55 channels)
- **Backbone**: `facebook/dinov2-large` (1024-dim self-supervised visual patch features)
- **4-Panel Telemetry-Command Dashboard**: Channel 0 Analog Telemetry (+-3 sigma bounds) + Command Activation Context + Volatility + Deviation Energy
- **Multi-Regime Operational Centroids**: $K=5$ KMeans nominal operating clusters from 58,317 clean train steps
- **Rendering**: Fast direct Agg canvas buffer copy (zero PNG compression bottleneck)
- **Multi-GPU & Batching**: Shards candidate windows across `cuda:0` and `cuda:1` with batch size 16

In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/VisTAD'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained VisTAD codebase (src, vlm4ts, configs)...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), 'r') as z:
    z.extractall(repo_dir)
print('VisTAD codebase unpacked successfully.')
print('Extracted top-level dirs:', os.listdir(repo_dir))


In [ ]:
# --- Step 2: GPU Accelerator & Multi-GPU Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} GPU(s):')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)')


In [ ]:
# --- Step 3: MSL Dataset Auto-Discovery (Direct Input Mount) ---
import os, glob, pandas as pd

c_tr = glob.glob('/kaggle/input/**/msl_train.csv', recursive=True)
c_te = glob.glob('/kaggle/input/**/msl_test.csv', recursive=True)
if c_tr and c_te:
    msl_dir = os.path.dirname(c_tr[0])
    print(f'Found MSL data directory: {msl_dir}')
    print(f'Train path: {c_tr[0]}')
    print(f'Test path:  {c_te[0]}')
else:
    raise FileNotFoundError('Could not locate msl_train.csv or msl_test.csv in /kaggle/input!')

df_tr = pd.read_csv(c_tr[0])
df_te = pd.read_csv(c_te[0])
print(f'MSL Train shape: {df_tr.shape}, MSL Test shape: {df_te.shape}')
print(f'MSL Test anomalies: {df_te["attack"].sum()} / {len(df_te)} ({100*df_te["attack"].mean():.2f}%)')


In [ ]:
# --- Step 4: Run Ultra-VETime MSL Cache Generation ---
import sys
sys.path.insert(0, '/kaggle/working/VisTAD')

from vlm4ts.generate_msl_cache import main

sys.argv = [
    'generate_msl_cache.py',
    '--data_dir', msl_dir,
    '--output_train_cache', '/kaggle/working/msl_train_ultravetime_cache.pt',
    '--output_eval_cache', '/kaggle/working/msl_eval_head_to_head_cache.pt',
    '--num_train_windows', '3000',
    '--num_eval_windows', '1500',
    '--model_id', 'facebook/dinov2-large',
    '--n_regimes', '5'
]

print('Starting MSL cache generation...')
main()
print('MSL cache generation completed successfully!')


In [ ]:
# --- Step 5: Clean Codebase and Verify Exported Caches ---
import os, torch, shutil

# Clean extracted codebase so only final caches remain in /kaggle/working
if os.path.exists('/kaggle/working/VisTAD'):
    shutil.rmtree('/kaggle/working/VisTAD')

train_c = '/kaggle/working/msl_train_ultravetime_cache.pt'
eval_c = '/kaggle/working/msl_eval_head_to_head_cache.pt'

for p in [train_c, eval_c]:
    if os.path.exists(p):
        sz = os.path.getsize(p) / (1024**2)
        dat = torch.load(p, map_location='cpu', weights_only=False)
        n_wins = len(dat.get('windows', dat))
        print(f'File: {os.path.basename(p)} | Size: {sz:.2f} MB | Windows: {n_wins}')
    else:
        print(f'MISSING: {p}')

print('Final Kaggle working directory contents:', os.listdir('/kaggle/working'))
